# 04.R2 — ResNet-50 Image Preprocessing & Training-Only Augmentation

**Phase 04 — Document Processing & OCR (ResNet-50 Visual Pipeline)**  
**Objective:** Define consistent ImageNet preprocessing (`224x224`, RGB, mean/std normalization) for all splits and **training-only** geometric/photometric augmentation (never applied to validation or test splits). Serializes the preprocessing specification to `models/phase_04/resnet50/preprocessing_config.json`.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
import torch
from torchvision import transforms

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

MANIFEST_CSV = paths.interim_doc_proc / 'resnet50' / 'resnet50_dataset_manifest.csv'
PREP_CFG_JSON = paths.resnet50_model_dir / 'preprocessing_config.json'
paths.resnet50_model_dir.mkdir(parents=True, exist_ok=True)

PREPROCESSING_SPEC = {
    'image_size': [224, 224],
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std': [0.229, 0.224, 0.225],
    'train_augmentation': {
        'random_rotation_deg': 5,
        'color_jitter': {'brightness': 0.15, 'contrast': 0.15},
        'random_affine_translate': [0.03, 0.03],
    },
    'val_test_augmentation': 'NONE (deterministic resize + normalize only)',
}

train_transform = transforms.Compose([
    transforms.Resize(tuple(PREPROCESSING_SPEC['image_size'])),
    transforms.RandomRotation(degrees=PREPROCESSING_SPEC['train_augmentation']['random_rotation_deg']),
    transforms.ColorJitter(**PREPROCESSING_SPEC['train_augmentation']['color_jitter']),
    transforms.RandomAffine(degrees=0, translate=tuple(PREPROCESSING_SPEC['train_augmentation']['random_affine_translate'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=PREPROCESSING_SPEC['imagenet_mean'], std=PREPROCESSING_SPEC['imagenet_std']),
])

eval_transform = transforms.Compose([
    transforms.Resize(tuple(PREPROCESSING_SPEC['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=PREPROCESSING_SPEC['imagenet_mean'], std=PREPROCESSING_SPEC['imagenet_std']),
])

with open(PREP_CFG_JSON, 'w', encoding='utf-8') as f:
    json.dump(PREPROCESSING_SPEC, f, indent=2)

print(f"[OK] Saved preprocessing spec -> {PREP_CFG_JSON}")
if MANIFEST_CSV.exists():
    df_m = pd.read_csv(MANIFEST_CSV)
    sample_row = df_m[df_m['split'] == 'train'].iloc[0]
    img_path = paths.root / sample_row['relative_path']
    pil_im = ImageOps.exif_transpose(Image.open(img_path)).convert('RGB')
    t_eval = eval_transform(pil_im)
    t_aug = train_transform(pil_im)
    print(f"Verified tensor shape: eval={tuple(t_eval.shape)}, train_aug={tuple(t_aug.shape)}")

    mean = np.array(PREPROCESSING_SPEC['imagenet_mean']).reshape(3, 1, 1)
    std = np.array(PREPROCESSING_SPEC['imagenet_std']).reshape(3, 1, 1)
    def unnorm(t):
        return np.clip((t.numpy() * std + mean).transpose(1, 2, 0), 0.0, 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(unnorm(t_eval)); axes[0].set_title('Deterministic Eval Transform (Val/Test)'); axes[0].axis('off')
    axes[1].imshow(unnorm(t_aug)); axes[1].set_title('Train-Only Augmented Sample'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()
